# IngredientMassEstimation V4 — MLP especializados + masa total auxiliar

Primero diagnosticar **p × c de V3 usando archivos guardados**. Luego probar las nuevas ramas por separado.
V2/V3 se conservan: este notebook no entrena ni evalúa hasta habilitar los controles.
El modelo sigue recibiendo solo RGB; reutiliza etiquetas y mapas de profundidad 320 de V3.

V4 no reescala los gramos con la masa auxiliar. No cargar `last.pt` de V3 como resume.


## 1. Montar Drive

Si la carpeta está compartida, usar un acceso directo en Mi unidad o adaptar DRIVE en el punto 3.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, zipfile, hashlib
from IPython.display import display, HTML
from google.colab import drive
drive.mount("/content/drive")


## 2. Descargar la rama V4 publicada

Publicar primero la rama indicada. Repositorio privado: PRIVATE_REPO=True y secreto GITHUB_TOKEN. No se imprime ni guarda la credencial.


In [ ]:
URL = "https://github.com/marcogloazzo/IngredientMassEstimation.git"
BRANCH = "v4-specialized-heads"
PRIVATE_REPO = False
REPO = Path("/content/nutrition-code-v4")

def git_private(command):
    environment = os.environ.copy()
    try:
        if PRIVATE_REPO:
            from google.colab import userdata
            import base64
            token = userdata.get("GITHUB_TOKEN")
            authorization = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            environment.update({
                "GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader",
                "GIT_CONFIG_VALUE_0": f"Authorization: Basic {authorization}",
            })
            del token, authorization
        result = subprocess.run(command, env=environment, capture_output=True, text=True)
        if result.returncode:
            raise RuntimeError("Git falló: revisá URL, rama publicada y permisos del token. No se imprime la credencial.")
        return result.stdout.strip()
    finally:
        environment.clear()

if (REPO / ".git").is_dir():
    current = git_private(["git", "-C", str(REPO), "branch", "--show-current"])
    assert current == BRANCH, f"Esta carpeta tiene otra rama: {current}. Usá una carpeta nueva."
    git_private(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH])
else:
    git_private(["git", "clone", "--branch", BRANCH, "--single-branch", URL, str(REPO)])

PROJECT = next((p for p in (REPO, REPO / "IngredientMassEstimation")
                if (p / "config/query2label_100_v4.json").is_file()), None)
assert PROJECT is not None, "Falta publicar el código V4."
os.chdir(PROJECT)
print("Proyecto:", PROJECT)
print("Rama:", git_private(["git", "-C", str(REPO), "branch", "--show-current"]))

print("Commit:", git_private(["git", "-C", str(REPO), "rev-parse", "HEAD"]))


## 3. Dependencias y controles del experimento

Primero **split_only**, después **aux_only** y finalmente **combined**, cada uno en su carpeta. Así se separa el aporte de cada cambio. Mantener el mismo presupuesto, split y seed que V3.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-v3.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps"], check=True)
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DRIVE = Path("/content/drive/MyDrive/IngredientMassEstimation")
SOURCE_V3 = DRIVE / "v3"
V4 = DRIVE / "v4"
VARIANT = "split_only"  # split_only / aux_only / combined
RUN_TAG = "run_01"
REPORT_TAG = "eval_01"
DIAGNOSTIC_TAG = "epoch39_01"
configs = {"split_only": "query2label_100_v4_split_only.json",
           "aux_only": "query2label_100_v4_aux_only.json",
           "combined": "query2label_100_v4.json"}
assert VARIANT in configs
assert all("/" not in tag and "\\" not in tag and tag not in ("", ".", "..")
           for tag in (RUN_TAG, REPORT_TAG, DIAGNOSTIC_TAG))
CONFIG = PROJECT / "config" / configs[VARIANT]
RUN = V4 / "runs" / f"{VARIANT}_{RUN_TAG}"
VALIDATION = V4 / "reports" / f"{VARIANT}_{RUN_TAG}_{REPORT_TAG}" / "validation"
TEST = VALIDATION.parent / "test"
CHECKPOINT = VALIDATION / "calibrated.pt"
V3_VALIDATION = SOURCE_V3 / "reports/run_01/validation"
GATE_DIAGNOSTIC = V4 / "diagnostics" / f"v3_gate_{DIAGNOSTIC_TAG}"
LABELS_PARENT = Path("/content/v4_prepared")
PROCESSED = LABELS_PARENT / "processed_100_grouped_provisional"
DATA_PARENT = Path("/content/nutrition5k_v3")
DEPTH_CACHE = Path("/content/depth_v3_320")  # compatible V3 cache, no changes
DEPTH_ARCHIVE = SOURCE_V3 / "depth_v3_320.zip"

def run_stream(arguments):
    command = [str(arg) for arg in arguments]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    finally:
        process.stdout.close()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)

print("Device:", DEVICE, "| variante:", VARIANT)
print("RUN V4:", RUN)
print("Fuente V3 (solo lectura):", V3_VALIDATION)


## 4. Restaurar únicamente las etiquetas V3 para el diagnóstico

No regenerar el split ni el vocabulario. Se admite el ZIP creado en Windows con barras invertidas. Verificar sus hashes.


In [ ]:
from pathlib import PurePosixPath
import stat

def extract_zip_safe(archive, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as source:
        checked = []
        for info in source.infolist():
            name = info.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if relative.is_absolute() or not relative.parts or ":" in relative.parts[0]:
                raise ValueError(f"Nombre inseguro en ZIP: {name}")
            target = (destination / relative).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f"Ruta insegura en ZIP: {name}")
            if stat.S_ISLNK(info.external_attr >> 16):
                raise ValueError(f"No se aceptan enlaces en ZIP: {name}")
            checked.append((info, target, name.endswith("/")))
        for info, target, is_directory in checked:
            if is_directory:
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(info) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)


labels_zip = SOURCE_V3 / "data/processed_100_grouped_provisional.zip"
assert labels_zip.is_file(), f"Falta el paquete existente de V3: {labels_zip}"
if not (PROCESSED / "records.jsonl").is_file():
    extract_zip_safe(labels_zip, LABELS_PARENT)
package = json.loads((LABELS_PARENT / "package_manifest.json").read_text())
for name, digest in package["file_sha256"].items():
    path = PROCESSED / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"Etiquetas modificadas: {name}"
print("Etiquetas V3 conservadas:", PROCESSED)


## 5. Diagnóstico p × c de V3 — sin GPU ni modelo

Poner RUN_GATE_DIAGNOSTIC=True. Lee el NPZ de validación de la época seleccionada; no necesita dataset RGB, depth ni checkpoints. Elegir otro DIAGNOSTIC_TAG para repetir sin sobrescribir. Los mejores umbrales son exploratorios sobre esta misma validación, no mejoras demostradas en test.


In [ ]:
RUN_GATE_DIAGNOSTIC = False
if RUN_GATE_DIAGNOSTIC:
    assert (V3_VALIDATION / "predictions.npz").is_file(), "Falta predictions.npz de validación V3."
    run_stream([sys.executable, "-u", "scripts/diagnose_mass_gate.py",
                "--predictions", V3_VALIDATION / "predictions.npz",
                "--details", V3_VALIDATION / "evaluation_details.json",
                "--vocabulary", PROCESSED / "vocabulary.json",
                "--output-dir", GATE_DIAGNOSTIC])
if (GATE_DIAGNOSTIC / "diagnostic.html").is_file():
    display(HTML((GATE_DIAGNOSTIC / "diagnostic.html").read_text(encoding="utf-8")))
else:
    print("Diagnóstico desactivado: habilitar RUN_GATE_DIAGNOSTIC.")


## 6. Restaurar RGB y reutilizar depth V3 antes de entrenar

No hace falta volver a ejecutar Depth Anything: V4 mantiene la geometría/cache de V3. Esta celda solo restaura archivos y, opcionalmente, verifica datos; no carga el modelo.


In [ ]:
def dataset_roots():
    if not DATA_PARENT.is_dir():
        return []
    return sorted({folder.parent for folder in DATA_PARENT.rglob("metadata")
                   if folder.is_dir()
                   and (folder / "dish_metadata_cafe1.csv").is_file()
                   and (folder / "ingredients_metadata.csv").is_file()
                   and (folder.parent / "dish_ids").is_dir()
                   and (folder.parent / "imagery/realsense_overhead").is_dir()})

roots = dataset_roots()
if not roots:
    source = DRIVE / "nutrition5k_rgb.zip"
    assert source.is_file(), f"Falta: {source}"
    local = Path("/content/nutrition5k_rgb_v4.zip")
    shutil.copy2(source, local)
    extract_zip_safe(local, DATA_PARENT)
    roots = dataset_roots()
assert len(roots) == 1, f"Raíz RGB no única: {roots}"
DATA = roots[0]
if not (DEPTH_CACHE / "manifest.json").is_file():
    assert DEPTH_ARCHIVE.is_file(), "Falta el ZIP de mapas V3 320x320 en Drive."
    local = Path("/content/depth_v3_320.zip")
    shutil.copy2(DEPTH_ARCHIVE, local)
    extract_zip_safe(local, DEPTH_CACHE.parent)
assert (DEPTH_CACHE / "manifest.json").is_file(), "Revisá la estructura del ZIP de depth V3."
RUN_DATA_CHECK = False
if RUN_DATA_CHECK:
    run_stream([sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
                "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
                "--device", DEVICE, "--check-data"])
print("RGB:", DATA, "| cache compatible:", DEPTH_CACHE)


## 7. Entrenamiento V4 — desactivado

Habilitar **solo cuando decidas entrenar**. Empezar con split_only; no cambiar VARIANT después de configurar rutas.
La masa sigue siendo p × c. Aux_only/combined añaden pérdidas de masa auxiliar (0,25) y consistencia (0,10), normalizadas por la media train, no por una constante histórica.

MODE="resume" solo admite last.pt de esta misma variante. Cada variante mantiene su history.csv, best.pt, last.pt y predicciones de la mejor validación. Máximo 100 épocas, patience 8; no se entrenó V4 todavía.


In [ ]:
RUN_TRAINING = False
MODE = "new"
TOTAL_EPOCHS = 100
if RUN_TRAINING:
    assert (GATE_DIAGNOSTIC / "diagnostic.json").is_file(), "Primero ejecutar el diagnóstico del punto 5."
    assert MODE in ("new", "resume")
    args = [sys.executable, "-u", "scripts/train_image_model.py", "--config", CONFIG,
            "--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
            "--output-dir", RUN, "--torch-cache", DRIVE / "torch_cache",
            "--device", DEVICE, "--workers", "2", "--epochs", TOTAL_EPOCHS]
    if MODE == "resume":
        assert (RUN / "last.pt").is_file(), "Falta last.pt de esta variante V4."
        args += ["--resume", RUN / "last.pt"]
    else:
        assert not RUN.exists() or not any(RUN.iterdir()), "RUN tiene resultados: usar resume de V4 o un RUN_TAG nuevo."
    run_stream(args)
else:
    print("Entrenamiento V4 desactivado.")


## 8. Validación, calibración y test — desactivados

Evaluar las variantes primero en validación. Test se reserva para la arquitectura elegida. La calibración de presencia no cambia gramos. REPORT_TAG debe ser nuevo para repetir evaluaciones.


In [ ]:
RUN_CALIBRATION = False
RUN_TEST = False
shared = ["--dataset-root", DATA, "--processed-dir", PROCESSED, "--depth-cache", DEPTH_CACHE,
          "--device", DEVICE, "--batch-size", "2", "--workers", "2"]
if RUN_CALIBRATION:
    assert (RUN / "best.pt").is_file(), "Falta best.pt de esta V4."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", RUN / "best.pt",
                "--split", "validation", "--calibrate", "--output-dir", VALIDATION, *shared])
if RUN_TEST:
    assert CHECKPOINT.is_file(), "Primero calibrar con validación."
    run_stream([sys.executable, "-u", "scripts/evaluate_image_model.py", "--checkpoint", CHECKPOINT,
                "--split", "test", "--output-dir", TEST, *shared])
for split, folder in (("VALIDACIÓN", VALIDATION), ("TEST · solo referencia", TEST)):
    if (folder / "metrics.json").is_file():
        print(split)
        display(json.loads((folder / "metrics.json").read_text()))


## 9. Diagnóstico V4 y comparación controlada con V3 — archivos guardados

Comparar el mismo split/vocabulario. Observar L1 de ingredientes, MAE presente, falsos gramos, masa total y nutrientes. En aux_only/combined, revisar también MAE de masa auxiliar y su discrepancia con la suma de ingredientes; mejorar la masa auxiliar sola no demuestra mejorar el resultado final.


In [ ]:
RUN_V4_DIAGNOSTIC = False
RUN_COMPARISON = False
V4_DIAGNOSTIC = V4 / "diagnostics" / f"{VARIANT}_{RUN_TAG}_{REPORT_TAG}"
COMPARISON = V4 / "comparisons" / f"v3_vs_{VARIANT}_{RUN_TAG}_{REPORT_TAG}"
if RUN_V4_DIAGNOSTIC:
    run_stream([sys.executable, "-u", "scripts/diagnose_mass_gate.py",
                "--predictions", VALIDATION / "predictions.npz",
                "--details", VALIDATION / "evaluation_details.json",
                "--vocabulary", PROCESSED / "vocabulary.json", "--output-dir", V4_DIAGNOSTIC])
if (V4_DIAGNOSTIC / "diagnostic.html").is_file():
    display(HTML((V4_DIAGNOSTIC / "diagnostic.html").read_text(encoding="utf-8")))
if RUN_COMPARISON:
    run_stream([sys.executable, "-u", "scripts/compare_image_experiments.py",
                "--left", V3_VALIDATION, "--right", VALIDATION, "--output-dir", COMPARISON])
if (COMPARISON / "comparison.html").is_file():
    display(HTML((COMPARISON / "comparison.html").read_text(encoding="utf-8")))


## Git y Drive

**Git:** nueva rama v4-specialized-heads, código, 3 configs V4, scripts, tests, documentos y este notebook sin salidas. No hay commit/push automático.

**Drive:** conservar v2/ y v3/ completas. Reutilizar nutrition5k_rgb.zip, v3/data/processed_100_grouped_provisional.zip, v3/depth_v3_320.zip y v3/reports/run_01/validation/{predictions.npz,evaluation_details.json}. No subir nuevamente datos/pesos ya presentes. Abrir este notebook en Colab; las salidas nuevas van exclusivamente a v4/.

**Importante:** V4 necesita una corrida nueva; no es una continuación con resume del checkpoint V3. No se cambia a ASL, no se agrega Self-Attention ni se congelan etapas adicionales en este experimento.
